# Table 1

### Produce a table with all baseline models, 'monthly', with columns 

|full_cum_return|full_cagr|full_annual_volatility|full_sharpe|full_sortino|full_calmar|full_ulcer|full_max_dd|full_turnover_pa|`

In [ ]:
'''
Table 1: all baseline models, monthly, with the columns specified above.
RUNS: output folder name (_output/<model>/<folder>) -> name in the table, in
table order.  Values are read from the overall sheet of
crisis_analysis_all_<FREQ>.xlsx (crisis_analysis_defined.py), as decimals.
'''

import sys
from pathlib import Path

import pandas as pd

PAPER = Path.cwd().parent
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import crisis_analysis_defined as cad

RUNS = {
    'mcap_t_10_monthly':              'MCap',
    'equal_t_10_monthly':             'Equal Weight',
    'mvo_t_10_lw_monthly':            'MVO',
    'hrp_t_10_lw_ward_monthly':       'HRP',
    'xgb_t_10_holdout_monthly':       'XGB',
    'rf_t_10_holdout_monthly':        'RF',
    'lstm_t_10_holdout_rank_monthly': 'LSTM',
}
COLUMNS  = ['full_cum_return', 'full_cagr', 'full_annual_volatility', 'full_sharpe', 'full_sortino',
            'full_calmar', 'full_ulcer', 'full_max_dd', 'full_turnover_pa']
FREQ     = 'monthly'
OUT_PATH = Path.cwd() / 'data_finalized' / 'table_1.csv'

# table column -> source column, where the two differ
ALIASES = {'full_annual_volatility': 'full_ann_vol'}

src = pd.read_excel(cad.SAVE_DIR / f'crisis_analysis_all_{FREQ}.xlsx', sheet_name='overall')
src = src[src['frequency'] == FREQ].set_index('run')

# folder name = run name + '_<FREQ>'
runs = {folder.removesuffix(f'_{FREQ}'): label for folder, label in RUNS.items()}
missing = [r for r in runs if r not in src.index]
if missing:
    raise KeyError(f'{missing} not in the {FREQ} overall sheet; runs there: {src.index.tolist()}')

table_1 = src.loc[list(runs), [ALIASES.get(c, c) for c in COLUMNS]]
table_1.columns = COLUMNS
table_1.insert(0, 'model', list(runs.values()))

OUT_PATH.parent.mkdir(exist_ok=True)
table_1.to_csv(OUT_PATH, index=False)
print(f'wrote {OUT_PATH}')
table_1

wrote c:\Users\benel\Coding\Python\Paper\_latex\data_finalized\table_1.csv


,model,full_cum_return,full_cagr,full_annual_volatility,full_sharpe,full_sortino,full_calmar,full_ulcer,full_max_dd,full_turnover_pa
run,,,,,,,,,,
mcap_t_10,MCap,15.230356,0.104872,0.203124,0.387720,0.544352,0.200391,0.185109,-0.523337,0.161855
equal_t_10,Equal Weight,13.595801,0.100683,0.197016,0.380466,0.533320,0.181004,0.144370,-0.556246,0.766384
mvo_t_10_lw,MVO,12.773208,0.098400,0.193538,0.376570,0.528831,0.182025,0.202196,-0.540588,2.089923
hrp_t_10_lw_ward,HRP,11.342476,0.094098,0.171722,0.401557,0.565282,0.206447,0.104903,-0.455797,1.295281
xgb_t_10_holdout,XGB,14.773450,0.103743,0.216748,0.358647,0.503508,0.168476,0.154927,-0.615775,5.494242
rf_t_10_holdout,RF,15.337051,0.105131,0.219495,0.359883,0.504866,0.172346,0.159873,-0.610000,4.829376
lstm_t_10_holdout_rank,LSTM,15.946816,0.106581,0.218367,0.367744,0.515433,0.173398,0.166725,-0.614662,3.360186


# Table 2

### Produce a table with the GSPC (first row) and all baseline models, 'monthly', with per crisis C1 ... C7 the columns

|cx_full_return|cx_full_max_dd|

In [ ]:
'''
Table 2: return and maximum drawdown per crisis (peak -> recovery, crises.py),
GSPC in the first row, then the baseline runs of Table 1.
Models: sheets C1 ... C7 of crisis_analysis_all_<FREQ>.xlsx.  GSPC is not in
that workbook (crisis_analysis_defined skips sp500), so its row is computed
here from the sp500 run with the same window_row() the workbook uses.
'''

import sys
from pathlib import Path

import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
import crisis_analysis_defined as cad

FREQ = 'monthly'
# baseline runs (run name -> name in the table), as in Table 1
RUNS_2 = {
    'mcap_t_10':              'MCap',
    'equal_t_10':             'Equal Weight',
    'mvo_t_10_lw':            'MVO',
    'hrp_t_10_lw_ward':       'HRP',
    'xgb_t_10_holdout':       'XGB',
    'rf_t_10_holdout':        'RF',
    'lstm_t_10_holdout_rank': 'LSTM',
}
OUT_PATH_2 = PAPER / '_latex' / 'data_finalized' / 'table_2.csv'
METRICS_2  = {'full_cum_return': 'full_return', 'full_max_dd': 'full_max_dd'}  # source -> table
crises     = [name for name, _, phases in cad.WINDOWS if phases]               # C1 ... C7

sheets = pd.read_excel(cad.SAVE_DIR / f'crisis_analysis_all_{FREQ}.xlsx', sheet_name=crises)

# GSPC: sp500 run of the same frequency, benchmark irrelevant for these two metrics
sp = next(r for r in ca.find_runs() if r['model'] == 'sp500' and r['frequency'] == FREQ)
gspc = {}
for name, w, _ in cad.WINDOWS:
    if name in crises:
        row = cad.window_row(sp, sp['returns'], w, phases=True)
        gspc.update({f'{name.lower()}_{t}': row[s] for s, t in METRICS_2.items()})

rows = [pd.Series(gspc, name='GSPC')]
for run, label in RUNS_2.items():
    vals = {}
    for c in crises:
        src = sheets[c]
        hit = src[(src['run'] == run) & (src['frequency'] == FREQ)]
        if hit.empty:
            raise KeyError(f'{run!r} not in the {FREQ} {c} sheet')
        vals.update({f'{c.lower()}_{t}': hit.iloc[0][s] for s, t in METRICS_2.items()})
    rows.append(pd.Series(vals, name=label))

table_2 = pd.DataFrame(rows).rename_axis('model').reset_index()
table_2.to_csv(OUT_PATH_2, index=False)
print(f'wrote {OUT_PATH_2}: {table_2.shape[0]} rows x {table_2.shape[1] - 1} columns')
table_2

# Table 3

### Borda ranking of the Sharpe ratio, 37 monthly runs x 5 columns

Rows: MCap, Equal Weight, then per model (MVO, HRP, XGB, LSTM, RF) the baseline and changepoint / HMM / Wasserstein with cash and with the alternative (min-var for MVO and HRP, HRP for the ML models).

|full_sharpe|crisis_equal|crisis_drawdown|crisis_duration|crisis_vix|

full: Sharpe 1998-2025.  crisis_*: sum over C1 ... C7 of weight x crisis Sharpe (peak -> recovery); weights equal, S&P 500 max drawdown, duration peak -> recovery, mean VIX.  Each column: rank over the 37 runs from the Borda count (1 = best, 37 = worst, ties share the average rank).

In [ ]:
'''
Table 3: Borda ranking of the Sharpe ratio for 37 monthly runs (spec above).
Sharpe ratios as in compare_all.window_metrics (metrics.sharpe_ratio over
the returns in (start, end]).  Crisis weights: equal, drawdown and VIX from
the weights sheet of z_scores_monthly_base.xlsx (crisis_severity.py);
duration = calendar days peak -> recovery from crises.py, normalised to 1.
'''

import sys
from pathlib import Path

import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
from crises import overall, main_crises
from metrics import sharpe_ratio

FREQ         = 'monthly'
OUT_PATH_3   = PAPER / '_latex' / 'data_finalized' / 'table_3_borda_sharpe.csv'
WEIGHTS_XLSX = PAPER / '_output' / '_comparison' / 'z_scores_monthly_base.xlsx'

# model: (name in the table, baseline run, alternative crisis target, its name)
MODELS = {
    'mvo':  ('MVO',  'mvo_t_10_lw',            'min_var', 'min-var'),
    'hrp':  ('HRP',  'hrp_t_10_lw_ward',       'min_var', 'min-var'),
    'xgb':  ('XGB',  'xgb_t_10_holdout',       'hrp',     'HRP'),
    'lstm': ('LSTM', 'lstm_t_10_holdout_rank', 'hrp',     'HRP'),
    'rf':   ('RF',   'rf_t_10_holdout',        'hrp',     'HRP'),
}
DETECTORS = {'changepoint_vix': 'CP', 'hmm_gspc': 'HMM', 'wasserstein_gspc_vix': 'WS'}

# name in the table -> run name, in table order
RUNS_3 = {'MCap': 'mcap_t_10', 'Equal Weight': 'equal_t_10'}
for label, base, alt, alt_label in MODELS.values():
    RUNS_3[label] = base
    for tag, det in DETECTORS.items():
        RUNS_3[f'{label} {det} cash']        = f'{base}_{tag}_cash'
        RUNS_3[f'{label} {det} {alt_label}'] = f'{base}_{tag}_{alt}'

# ---- crisis weights (C1 ... C7)
crisis_ids = [f'C{i}' for i in range(1, len(main_crises) + 1)]
w_sheet = pd.read_excel(WEIGHTS_XLSX, sheet_name='weights', index_col=0)
if list(w_sheet.columns) != crisis_ids:
    raise ValueError(f'weights sheet has {list(w_sheet.columns)}, crises.py has {crisis_ids}: rerun z_scores.py')
days = pd.Series([(pd.Timestamp(c['even']) - pd.Timestamp(c['peak'])).days for c in main_crises],
                 index=crisis_ids)
WEIGHTS = {
    'crisis_equal':    w_sheet.loc['Equal weight'],
    'crisis_drawdown': w_sheet.loc['Drawdown weight'],
    'crisis_duration': days / days.sum(),
    'crisis_vix':      w_sheet.loc['VIX weight'],
}

# ---- Sharpe ratios: full period and per crisis
by_run = {r['run']: r for r in ca.find_runs() if r['frequency'] == FREQ}
missing = [run for run in RUNS_3.values() if run not in by_run]
if missing:
    raise KeyError(f'no {FREQ} run for {missing}')

sharpe = pd.DataFrame(
    {label: [sharpe_ratio(ca._slice(by_run[run]['returns'], overall[0]['peak'], overall[0]['even']))]
            + [sharpe_ratio(ca._slice(by_run[run]['returns'], c['peak'], c['even'])) for c in main_crises]
     for label, run in RUNS_3.items()},
    index=['full_sharpe'] + crisis_ids).T

scores = pd.DataFrame({'full_sharpe': sharpe['full_sharpe']})
for col, w in WEIGHTS.items():
    scores[col] = sharpe[crisis_ids].mul(w, axis=1).sum(axis=1)

# ---- rank per column from the Borda count: 1 = highest Sharpe, ties share the average rank
table_3 = scores.rank(ascending=False, method='average').rename_axis('model').reset_index()

OUT_PATH_3.parent.mkdir(exist_ok=True)
table_3.to_csv(OUT_PATH_3, index=False)
print(f'wrote {OUT_PATH_3}: {table_3.shape[0]} rows x {table_3.shape[1] - 1} columns')
print(pd.DataFrame(WEIGHTS).T.round(3).to_string())
table_3